In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

# 1. CARGA DE DATOS
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

def clean_data(df):
    # Copia para no modificar el original
    df = df.copy()
    
    # Limpieza de Ram: "8GB" -> 8 (int)
    if 'Ram' in df.columns:
        df['Ram'] = df['Ram'].str.replace('GB', '').astype(int)
    
    # Limpieza de Weight: "2.1kg" -> 2.1 (float)
    if 'Weight' in df.columns:
        df['Weight'] = df['Weight'].str.replace('kg', '').astype(float)
    
    # Ingeniería sencilla de CPU: Extraer la frecuencia (GHz)
    df['Cpu_GHz'] = df['Cpu'].str.extract(r'(\d+(?:\.\d+)?)GHz').astype(float)
    
    # Identificar si tiene SSD (binario)
    df['Has_SSD'] = df['Memory'].apply(lambda x: 1 if 'SSD' in x else 0)
    
    # Dropear columnas que no usaremos o que ya procesamos
    cols_to_drop = ['id', 'laptop_ID', 'Product', 'Cpu', 'Memory']
    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
    
    return df

# Aplicar limpieza
train_clean = clean_data(train_df)
test_clean = clean_data(test_df)

# 2. DEFINICIÓN DE FEATURES Y TARGET
X = train_clean.drop(columns=['Price_euros'])
y = train_clean['Price_euros']

# Identificar columnas numéricas y categóricas
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

# 3. PREPROCESAMIENTO (Pipeline)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])


C:\Users\NaiaJon\AppData\Local\Temp\ipykernel_7004\5881657.py:52: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include=['object']).columns.tolist()


Evaluando modelos (MAE)...
Linear Regression: MAE = 234.09
Polynomial Regression (Deg 2): MAE = 423.39
Decision Tree: MAE = 254.30
Random Forest: MAE = 205.14
AdaBoost: MAE = 383.13
Gradient Boosting: MAE = 222.28
XGBoost: MAE = 206.88

El mejor modelo es: Polynomial Regression (Deg 2) con MAE de 205.14


In [3]:
X.columns

Index(['Company', 'TypeName', 'Inches', 'ScreenResolution', 'Ram', 'Gpu',
       'OpSys', 'Weight', 'Cpu_GHz', 'Has_SSD'],
      dtype='str')

In [6]:

# 4. DEFINICIÓN DE MODELOS A PROBAR
models = {
    "Linear Regression": LinearRegression(),
    "Polynomial Regression (Deg 2)": Pipeline([
        ('poly', PolynomialFeatures(degree=2)),
        ('reg', LinearRegression())
    ]),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "AdaBoost": AdaBoostRegressor(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
}

# 5. ENTRENAMIENTO Y EVALUACIÓN (Cross-Validation)
best_model = None
best_mae = float('inf')
results = {}

print("Evaluando modelos (MAE)...")
for name, model in models.items():
    # Crear un pipeline que combine preprocesamiento + modelo
    clf = Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])
    
    # Usamos validación cruzada para mayor seguridad
    scores = cross_val_score(clf, X, y, cv=5, scoring='neg_mean_absolute_error')
    mae = -scores.mean()
    results[name] = mae
    print(f"{name}: MAE = {mae:.2f}")
    
    if mae < best_mae:
        best_mae = mae
        best_model = clf

# 6. ENTRENAMIENTO FINAL DEL MEJOR MODELO
print(f"\nEl mejor modelo es: {min(results, key=results.get)} con MAE de {best_mae:.2f}")


Evaluando modelos (MAE)...
Linear Regression: MAE = 234.09
Polynomial Regression (Deg 2): MAE = 423.39
Decision Tree: MAE = 254.30
Random Forest: MAE = 205.14
AdaBoost: MAE = 383.13
Gradient Boosting: MAE = 222.28
XGBoost: MAE = 206.88

El mejor modelo es: Random Forest con MAE de 205.14


In [4]:
# Esto te dirá el número de columnas finales tras la transformación
X_transformed = preprocessor.fit_transform(X)
print(f"Número total de features: {X_transformed.shape[1]}")

Número total de features: 170


In [2]:
best_model.fit(X, y)

# 7. PREDICCIÓN PARA SUBMISSIÓN
predictions = best_model.predict(test_clean)

submission = pd.DataFrame({
    'id': pd.read_csv('test.csv')['id'],
    'Price_euros': predictions
})

submission.to_csv('my_submission.csv', index=False)
print("\nArchivo 'my_submission.csv' generado con éxito.")


Archivo 'my_submission.csv' generado con éxito.
